# 4. Machine Learning\nEnd-to-end model training, evaluation, and inference workflow.

In [ ]:
from pathlib import Path\n\nimport pandas as pd\nfrom sklearn.metrics import accuracy_score, classification_report, precision_recall_curve, roc_auc_score\nfrom sklearn.model_selection import train_test_split\n\nfrom src.config.core import config\nfrom src.predict import make_prediction\nfrom src.processing.data_manager import load_dataset, load_pipeline, model_file_name\nfrom src.train_pipeline import run_training\n\nROOT = Path.cwd()\nif not (ROOT / 'src').exists():\n    ROOT = ROOT.parent

In [ ]:
train_metrics = run_training()\nprint('Training metrics:', train_metrics)\n\nmodel = load_pipeline(model_file_name())\nprint('Loaded model artifact:', model_file_name())

In [ ]:
data = load_dataset(config.app_config.training_data_file)\nx = data.drop(columns=[config.ml_config.target])\ny = data[config.ml_config.target].astype(int)\n\nx_train, x_valid, y_train, y_valid = train_test_split(\n    x, y,\n    test_size=config.ml_config.test_size,\n    random_state=config.ml_config.random_state,\n    stratify=y,\n)\n\npred = model.predict(x_valid)\nproba = model.predict_proba(x_valid)[:, 1]

In [ ]:
print('Validation Accuracy:', round(accuracy_score(y_valid, pred), 4))\nprint('Validation ROC AUC :', round(roc_auc_score(y_valid, proba), 4))\nprint(classification_report(y_valid, pred))

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_valid, proba)\nthreshold_frame = pd.DataFrame({\n    'threshold': list(thresholds) + [1.0],\n    'precision': precision,\n    'recall': recall,\n})\nthreshold_frame.head(10)

In [ ]:
test_df = pd.read_csv(ROOT / 'src/data/test.csv')\ntest_predictions = make_prediction(test_df)\n\npred_df = pd.DataFrame({\n    'enrollee_id': test_df['enrollee_id'],\n    'prediction': test_predictions['predictions'],\n})\nif 'probabilities' in test_predictions:\n    pred_df['probability'] = test_predictions['probabilities']\n\npred_df.head()

In [ ]:
output_dir = ROOT / 'notebooks/outputs'\noutput_dir.mkdir(parents=True, exist_ok=True)\noutput_path = output_dir / 'test_predictions.csv'\npred_df.to_csv(output_path, index=False)\nprint('Saved predictions to:', output_path)

In [ ]:
api_like_payload = {\n    'inputs': [\n        {\n            'city': 'city_41',\n            'city_development_index': 0.827,\n            'gender': 'Male',\n            'relevent_experience': 'Has relevent experience',\n            'enrolled_university': 'Full time course',\n            'education_level': 'Graduate',\n            'major_discipline': 'STEM',\n            'experience': '9',\n            'company_size': '<10',\n            'company_type': 'Pvt Ltd',\n            'last_new_job': '1',\n            'training_hours': 21.0\n        }\n    ]\n}\n\nmake_prediction(pd.DataFrame(api_like_payload['inputs']))

This notebook can be reused for regression checks after pipeline updates: retrain -> evaluate -> run batch inference -> export predictions.